In [60]:
"""
Notebook for performing background subtraction from original raw COMET ome.tiff files
the notebook uses the same Normalized Subtraction formula used by the HORIZON software (see user manual),
and the resulting images are equivalent. The original metadata are rewritten to the image to mantain
compatibility with the horizon software.
"""

import copy
import gc
import tifffile
import ome_types
import zarr
import cv2
import numpy as np
import dask.array as da
from unidecode import unidecode

In [164]:
# Raw COMET image
input_path = "./raw_comet_image.ome.tiff"

In [62]:
# Read the raw image as a dask array
comet_img = tifffile.imread(input_path, aszarr=True)
comet_img = zarr.open(comet_img, mode='r')
comet_img = da.from_zarr(comet_img["0"])
print(comet_img)

dask.array<from-zarr, shape=(19, 44643, 44643), dtype=uint16, chunksize=(1, 512, 512), chunktype=numpy.ndarray>

In [63]:
# Read the original metadata
with tifffile.TiffFile(input_path) as tif:
    ome_xml = tif.ome_metadata
ome = ome_types.from_xml(ome_xml)

In [64]:
# Extract the metadata for channels, planes and additional channel annotations ("children")
raw_channels = ome.images[0].pixels.channels 
raw_planes = ome.images[0].pixels.planes
raw_childrens = ome.structured_annotations[0].value.any_elements[0].children

## 5.7.3 Subtracting Background Signal
Background subtraction is used to remove background signals from the image. 
Background signal can have various sources such as tissue autofluorescence or non-specific staining.

The method available in HORIZON for background subtraction is Normalized Subtraction
which applies the following transformation to the pixels of the channel:
𝑃𝑆 = 𝑃𝑂 − (𝑃𝐵 × 𝐸𝑂) / 𝐸𝐵
With P the pixel value, E the exposure time, S the subtracted channel, O the original 
channel and B the background channel.

*From "Horizon and Viewer User Manual V5", page 19*

In [132]:
# Define the subtraction channels:
# {
#  "NEW_CHANNEL_NAME": ("RAW_CHANNEL", "BG_CHANNEL"|None)  
# }
# If the background channel is None, then the raw channel is rewritten as is
# Only channels mentioned in the dictionary are included in the output
# E.g. to include the original background channels add them in the dict with the
# background channel set to None
subtractions = {"DAPI":  ('DAPI_Cycle_001', None), 
                "CD3":   ('TRITC_Cycle_002', 'TRITC_Cycle_001'),
                "FOXP3": ('Cy5_Cycle_002', 'Cy5_Cycle_001'),
                "CD4":   ('TRITC_Cycle_004', 'TRITC_Cycle_003'),
                "CD8":   ('Cy5_Cycle_004', 'Cy5_Cycle_003'),
}

In [133]:
def get_channel_number(name, raw_channels):
    """
    Finds the index of a channel with a given name in the channel metadata
    """
    for i,c in enumerate(raw_channels):
        if name ==  c.name:
            return i
    return None

In [67]:
def get_exposure_time(channel_id, raw_planes):
    """
    Gets the exposure time of a channel from the plane metadata
    """
    return raw_planes[channel_id].exposure_time

In [100]:
def subtract_channels(name_original, name_background, img, raw_channels, raw_planes):
    """
    Subtracts two channels by name using the Normalized Subtraction method
    which applies the following transformation to the pixels of the channel:
    PS = PO - (PB * EO) / EB
    With P the pixel value, E the exposure time, S the subtracted channel, O the original channel and B the background channel.
    Values < 0 are clipped to 0.
    """
    id_original    = get_channel_number(name_original, raw_channels)
    id_background  = get_channel_number(name_background, raw_channels)
    exp_original   = get_exposure_time(id_original, raw_planes)
    exp_background = get_exposure_time(id_background, raw_planes)
    original   = img[id_original]
    background = img[id_background]
    cleaned = original - (background * exp_original) / exp_background
    cleaned *= (cleaned > 0) # Clip negative values to 0
    return cleaned

In [134]:
def build_subtracted_image(subtraction_channels, img, raw_channels, raw_planes):
    """
    Builds an image with channels with background subtraction accoring to the subtraction_channels
    dictionary which has the following format:
    {
      "NEW_CHANNEL_NAME": ("RAW_CHANNEL", "BG_CHANNEL"|None)  
    }
    If the background channel is None, then the raw channel is rewritten as is
    Only channels mentioned in the dictionary are included in the output
    E.g. to include the original background channels add them in the dict with the
    background channel set to None. The output is converted to uint16
    """
    subtracted_images = []
    for channlel_name, cycles in subtraction_channels.items():
        if not cycles[1]:
            cid = get_channel_number(cycles[0], raw_channels)
            subtracted_images.append(img[cid])
        else:
            c_img = subtract_channels(cycles[0], cycles[1], img, raw_channels, raw_planes)
            subtracted_images.append(c_img)
    return da.stack(subtracted_images).astype("uint16")

In [102]:
def build_metadata(subtraction_channels, raw_channels, raw_planes, raw_childrens):
    """
    Builds the channel, planes and "children" annotation metadata for the
    subtracted image using the metadata of the raw image as a starting point
    """
    
    new_channels = []
    new_planes = []
    new_children = []
    for n, it in enumerate(subtraction_channels.items()):
        channel = ome_types.model.Channel(id=f"Channel:{n}", name=it[0], samples_per_pixel=1)
        new_channels.append(channel)
        
        id_original = get_channel_number(it[1][0], raw_channels)
        plane = ome_types.model.Plane(the_z=0, the_t=0, the_c=n,
                                      exposure_time= get_exposure_time(id_original, raw_planes),
                                      exposure_time_unit='ms')
        new_planes.append(plane)
        
        child = raw_childrens[id_original]
        child.attributes['ID'] = f'Channel:{n}'
        new_children.append(child)
        
    return new_channels, new_planes, new_children

In [103]:
sub_img = build_subtracted_image(subtractions, comet_img, raw_channels, raw_planes)

In [144]:
new_channels, new_planes, new_children = build_metadata(subtractions, raw_channels, raw_planes, raw_childrens)

In [145]:
# Prepare the metadata of the subtracted image using a copy of the metadata from the raw image
new_meta = copy.deepcopy(ome)
new_meta.images[0].pixels.channels = new_channels
new_meta.images[0].pixels.planes = new_planes
new_meta.structured_annotations[0].value.any_elements[0].children = new_children
new_meta.images[0].pixels.size_c = sub_img.shape[0]

# Functions to write a pyramidal ome.tiff file from a dask array
# Adapted from palom:
# https://github.com/labsyspharm/palom/blob/main/palom/pyramid.py
# MIT License

Copyright (c) 2024 Laboratory of Systems Pharmacology @ Harvard

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

# Limitations
 The functions are stripped down to the esssentials for writing COMET images.  
 Assumptions: 
- We have enough memory to hold 1 whole channel (may not work for technologies with much larger images)
- It is OK to blur for downsampling (not suitable for masks or orher uses where exact values in the downsampled levels are required)
- The image has shape cyx (even when only opne channel is present)

In [146]:
def write_pyramid(
    img,
    output_path,
    pixel_size=0.28,
    channel_names=None,
    downscale_factor=2,
    num_levels=5,
    compression="zlib",
    tile_size=1024,
    kwargs_tifffile=None,
):
    """
    Writes a pyramidal ome.tiff file (metadata excluded). Assumptions: 
    - We have enough memory to hold 1 whole channel (may not work for technologies with much larger images)
    - It is OK to blur for downsampling (not suitable for masks or orher uses where exact values in the downsampled levels are required)
    - The image has shape cyx (even when only opne channel is present)
    """
    num_channels = img.shape[0]
    base_shape = img.shape[1:3]

    factors = downscale_factor ** np.arange(num_levels)
    shapes = np.ceil(np.array(base_shape) / factors[:, None])
    shapes = [tuple(map(int, s)) for s in shapes]
 
    if tile_size is not None:
        assert tile_size % 16 == 0, (
            f"tile_size must be None or multiples of 16, not {tile_size}"
        )
        tile_shapes = [(tile_size, tile_size)] * num_levels

    dtype = img.dtype
    pixel_size = pixel_size
   
    with tifffile.TiffWriter(output_path, bigtiff=True) as tif:
        if kwargs_tifffile is None:
            kwargs_tifffile = {}
        kwargs = {
            **kwargs_tifffile,
            **dict(
                compression=compression,
                resolutionunit="CENTIMETER",
                dtype=dtype,
            ),
        }
        tif.write(
            data=tile_from_img(
                img, tile_shape=tile_shapes[0]),
            shape=(num_channels, *shapes[0]),
            subifds=int(num_levels - 1),
            tile=tile_shapes[0],
            **{
                **kwargs,
                **{"resolution": (1e4 / pixel_size, 1e4 / pixel_size)},
            },
        )
        tif.filehandle.flush()
        for level, (shape, tile_shape) in enumerate(zip(shapes[1:], tile_shapes[1:])):
            mag = downscale_factor ** (level + 1)
            tif.write(
                data=tile_from_pyramid(
                    output_path,
                    num_channels,
                    tile_shape=tile_shape,
                    downscale_factor=downscale_factor,
                    level=level,
                ),
                shape=(num_channels, *shape),
                subfiletype=1,
                tile=tile_shape,
                **{
                    **kwargs,
                    **{"resolution": (1e4 / mag / pixel_size, 1e4 / mag / pixel_size)},
                },
            )
            tif.filehandle.flush()


def tile_from_img(img, tile_shape):
    """
    Tile generator for the level 0 (full resolution image)
    """
    num_rows, num_cols = img.shape[1:3]
    h, w = tile_shape
    for cidx in range(img.shape[0]):
        c = img[cidx].compute()
        for y in range(0, num_rows, h):
            for x in range(0, num_cols, w):
                yield np.array(c[y : y + h, x : x + w])
        c = None

def tile_from_pyramid(
    path,
    num_channels,
    tile_shape,
    downscale_factor=2,
    level=0,
):
    """
    Tile generator for levels 1-(num_levels-1) (downsampled images)
    Reads the image written for the previous level and downsamples it to generate the tiles.
    """
    for c in range(num_channels):
        gc.collect()
        img = da.from_zarr(
            zarr.open(
                tifffile.imread(path, series=0, level=level, aszarr=True), mode="r"
            ),
            name=False,
        )
        if img.ndim == 2:
            img = img.reshape(1, *img.shape)
        img = img[c]
        img = img.map_blocks(
            cv2.blur, ksize=(downscale_factor, downscale_factor), anchor=(0, 0)
        )
        img = img.compute()
        num_rows, num_columns = img.shape
        h, w = tile_shape
        h *= downscale_factor
        w *= downscale_factor
        for y in range(0, num_rows, h):
            for x in range(0, num_columns, w):
                yield np.array(
                    img[y : y + h : downscale_factor, x : x + w : downscale_factor]
                )
        img = None

In [147]:
output_path = "./cleaned_image.ome.tiff"

In [148]:
# Write 
write_pyramid(sub_img, output_path, channel_names=subtractions.keys())

In [149]:
# Convert back to XML and remove unsopported unicode chars
new_meta_xml = new_meta.to_xml()
new_meta_xml = unidecode(new_meta_xml)
# Overwrite metadata
tifffile.tiffcomment(output_path, new_meta_xml)